In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistota typu A 
def unc_A(data):
    '''
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

def dms(stupne):
    stupne = np.asarray(stupne, dtype=float)
    znamenko = np.sign(stupne)
    celkem = np.round(np.abs(stupne) * 3600, 2)

    m_celkem, s = np.divmod(celkem, 60)
    d, m = np.divmod(m_celkem, 60)

    return znamenko, d, m, s

In [ ]:
# nacteni google tabulek
sheet_id = '1WguDcMu3y95ozOn7a2cBLHqNAV9YbfoBmDYihw6zr6o'
gid = '0'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_hranol = pd.read_csv(url)
df_hranol.dropna()

In [ ]:
# lamavy uhel

psi1_df = df_hranol['ψ₁ – stupně'] + df_hranol['ψ₁ – minuty']/60 + df_hranol['ψ₁ – vteřiny']/3600
psi1 = psi1_df.to_numpy(dtype=float)

psi2_df = df_hranol['ψ₂ – stupně'] + df_hranol['ψ₂ – minuty']/60 + df_hranol['ψ₂ – vteřiny']/3600
psi2 = psi2_df.to_numpy(dtype=float)

omega = (180 - (psi1 - psi2))*np.pi/180


znamenko, d, m, s = dms(omega*180/np.pi)
for u, zn, dd, mm, ss in zip(omega*180/np.pi, znamenko, d, m, s):
    minus = "-" if zn < 0 else ""
    print(f'{minus}{int(dd)} {int(mm):02d} {ss:05.2f}')


omega_uf = uf(np.mean(omega), unc_A(omega))
print(omega_uf*180/np.pi)
omega_uf_n = omega_uf.n

In [ ]:
# nacteni google tabulek
sheet_id = '1WguDcMu3y95ozOn7a2cBLHqNAV9YbfoBmDYihw6zr6o'
gid = '37383797'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_deviace = pd.read_csv(url)
df_deviace = df_deviace.dropna()
df_deviace

In [ ]:
# deviace

phi1_df = df_deviace['φ₁ – stupně'] + df_deviace['φ₁ – minuty']/60 + df_deviace['φ₁ – vteřiny']/3600
phi1 = phi1_df.to_numpy(dtype=float)

phi2_df = df_deviace['φ₂ – stupně'] + df_deviace['φ₂ – minuty']/60 + df_deviace['φ₂ – vteřiny']/3600
phi2 = phi2_df.to_numpy(dtype=float)

delta_min = abs((phi1 - phi2)/2)*np.pi/180
delta_min_test = np.array([66, 66, 67, 69, 71, 74])*np.pi/180

# index lomu

n = np.sin((delta_min+omega_uf_n)/2)/np.sin(omega_uf_n/2)
print(n)

In [ ]:
# cauchy fit
vln_delka = df_deviace['λ [nm]'].to_numpy(dtype=float)
vln_delka_lispace = np.linspace(400, 650, 1000)


def cauchy(delka_VLN, A, B, C):
    x_func = 1e4 /(delka_VLN**2)
    return A + B*x_func + C*x_func**2

p0 = [1.6, 0.05, 0.0]
popt, pcov = curve_fit(cauchy, vln_delka, n, p0=p0)
A, B, C = popt
unc_A, unc_B, unc_C = np.sqrt(np.diag(pcov))
B_nm,  unc_B_nm = B * 1e4,  unc_B * 1e4
C_nm,  unc_C_nm = C * 1e8,  unc_C * 1e8
#print(A, B_nm, C_nm)
#print(unc_A, unc_B_nm, unc_C_nm)

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(r'Index lomu', fontsize=20)
ax.set_xlabel(r'$\lambda$ [nm]', fontsize=20)
ax.scatter(vln_delka, n, label='Naměřené hodnoty', s=50)
ax.plot(vln_delka_lispace, cauchy(vln_delka_lispace, *popt), color='red', linestyle='-',
    label=r'Fit funkcí: $A + B/\lambda^2 + C/\lambda^4$''\n'
    fr'$A = {uf(A, unc_A):.1uL}$''\n'
    fr'$B = {uf(B_nm, unc_B_nm):.1uL}\,\mathrm{{nm^2}}$''\n'
    fr'$C = {uf(C_nm, unc_C_nm):.1uL}\,\mathrm{{nm^4}}$')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'cauchy.pdf', dpi=300, bbox_inches='tight')
#np.max(((n - cauchy(vln_delka, *popt))/n)*100)

fraunhofer_d = cauchy(587.6, *popt)
fraunhofer_F = cauchy(486.1, *popt)
fraunhofer_C = cauchy(656.3, *popt)

ny_d = (fraunhofer_d - 1)/(fraunhofer_F - fraunhofer_C)
print(fraunhofer_d)
print(ny_d)